# LLM-шлюз и трейсы: клиентская сторона (части 1–5)

В кластере работают агенты занятия 4, а между ними и провайдерами LLM теперь стоит **шлюз LiteLLM**
(`http://llm.agents.test`). Все вызовы LLM шлюз пишет в **Langfuse** (`http://langfuse.agents.test`).

**Перед запуском:** применены части 0–5 по README и прописаны домены (`make hosts`).

## 1. Шлюз — это обычный OpenAI API

Для клиента LiteLLM выглядит как OpenAI: тот же SDK, другой `base_url`. Ключ — **master key**
администратора (`part_1_litellm/secret.yaml`); дальше выдадим себе отдельный ключ.

In [ ]:
import httpx
from openai import OpenAI

LLM_URL = "http://llm.agents.test"
MASTER_KEY = "sk-master-1234"                  # учебный; в проде его не раздают клиентам

admin = OpenAI(base_url=LLM_URL, api_key=MASTER_KEY)
print("модели шлюза:", [m.id for m in admin.models.list()])

reply = admin.chat.completions.create(model="cheap-llm",
                                      messages=[{"role": "user", "content": "Столица Чехии? Одним словом."}])
print(reply.choices[0].message.content, "| токенов:", reply.usage.total_tokens)

## 2. Свой виртуальный ключ — с ограничениями

Виртуальный ключ создаёт шлюз (не OpenRouter). Ему можно ограничить модели, бюджет и частоту запросов.
Отозвать его — не трогая ключ провайдера.

In [ ]:
headers = {"Authorization": f"Bearer {MASTER_KEY}"}
httpx.post(f"{LLM_URL}/key/delete", headers=headers, json={"key_aliases": ["student"]})   # если остался с прошлого раза

key = httpx.post(f"{LLM_URL}/key/generate", headers=headers, json={
    "key_alias": "student",
    "models": ["cheap-llm"],     # только дешёвая модель
    "max_budget": 0.05,          # не больше $0.05 за всё время
    "rpm_limit": 3,              # не больше 3 запросов в минуту
}).json()["key"]
print("мой ключ:", key[:10] + "…")

student = OpenAI(base_url=LLM_URL, api_key=key, max_retries=0)
reply = student.chat.completions.create(model="cheap-llm", messages=[{"role": "user", "content": "Привет! Одним словом."}])
print("cheap-llm →", reply.choices[0].message.content)

Модель, которой нет в ключе, — отказ. А частые запросы упираются в `rpm_limit`
(счётчик общий для обеих реплик LiteLLM — он в Redis).

In [ ]:
import openai

try:
    student.chat.completions.create(model="bank-llm", messages=[{"role": "user", "content": "Привет"}])
except openai.APIStatusError as error:
    print("bank-llm →", error.status_code, error.message[:120])

for i in range(4):
    try:
        student.chat.completions.create(model="cheap-llm", messages=[{"role": "user", "content": "1+1? Одной цифрой"}])
        print(f"запрос {i + 1}: ok")
    except openai.RateLimitError as error:
        print(f"запрос {i + 1}: {error.status_code} — лимит запросов в минуту")

## 3. Расходы — по ключам

Шлюз считает токены и деньги **по каждому ключу**. У агентов свои ключи (`make agent-keys`),
поэтому видно, кто сколько потратил.

In [ ]:
def spend_by_key() -> dict:
    keys = httpx.get(f"{LLM_URL}/key/list", headers=headers, params={"return_full_object": "true"}).json()["keys"]
    return {k["key_alias"]: k["spend"] for k in keys}


for alias, spend in spend_by_key().items():
    print(f"{alias:<10} ${spend:.6f}")

## 4. Агенты ходят в LLM через шлюз

Задаём вопрос риск-менеджеру (как в занятии 4) и смотрим, на сколько выросли расходы его ключа и ключа аналитика.
Риск-менеджер сам тратит немного — основную работу (SQL-запросы) делает аналитик.
Если агенты уже переключены на свою модель (часть 3), прирост — $0: у своей модели нет цены за токен
(шлюз всё равно считает токены — их видно в Langfuse и в админке).

In [ ]:
import sys

sys.path.insert(0, "images/agents")
from ask import ask                            # images/agents/ask.py — тот же клиент, что в занятии 4

RISK_URL = "http://risk.agents.test"

before = spend_by_key()
decision = await ask(RISK_URL, "Можно ли выдать счёту 2378 кредит 150 000 CZK на 24 месяца?")
print(decision["answer"])

In [ ]:
import time

time.sleep(10)                                 # шлюз записывает расходы в базу пачками, раз в несколько секунд
after = spend_by_key()
for alias in ("risk", "analyst"):
    print(f"{alias:<8} потратил за решение: ${after[alias] - before.get(alias, 0):.6f}")

## 5. Трейсы в Langfuse

Каждый вызов LLM, прошедший через шлюз, LiteLLM отправил в Langfuse: модель, токены, цена, задержка,
а в метаданных — **алиас ключа**, то есть какой агент спрашивал. Достанем последние вызовы через API
Langfuse (ключи проекта — из `part_4_langfuse/values.yaml`).

UI: http://langfuse.agents.test — логин `admin@agents.test`, пароль `admin12345`.

In [ ]:
LANGFUSE_URL = "http://langfuse.agents.test"
LANGFUSE_AUTH = ("pk-lf-bank-agents", "sk-lf-bank-agents")

time.sleep(10)                                 # события доходят до Langfuse через очередь — несколько секунд
observations = httpx.get(f"{LANGFUSE_URL}/api/public/v2/observations", auth=LANGFUSE_AUTH,
                         params={"limit": 15, "fields": "core,basic,usage,model,metadata"}).json()["data"]
for o in observations:
    agent = o["metadata"].get("user_api_key_alias") or "master"
    print(f"{o['startTime'][11:19]}  {agent:<8} {o['model']:<32} {o['totalUsage']:>6} ток.  "
          f"${o['totalCost'] or 0:.6f}  {o['latency']:.1f} с")

## Итоги

- агенты и любые клиенты ходят в LLM через **один адрес** — шлюз; провайдера и модель меняет шлюз, а не код агентов;
- у каждого клиента **свой виртуальный ключ**: доступ к моделям, бюджет, лимиты и учёт расходов;
- своя модель (vLLM на ноутбуке или ВМ с GPU) добавляется в шлюз как ещё одна модель, а на случай её падения есть **fallback**;
- все вызовы видны в **Langfuse** — без единой строчки кода в агентах.